## Attention from scratch

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

### Single Attention Head

In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, seq_len, d_k)
    K: (batch, seq_len, d_k)
    V: (batch, seq_len, d_v)
    """
    d_k     = Q.size(-1)
    scores  = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # scores shape: (batch, seq_len, seq_len)
    # Each row = how much one token attends to every other token

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
        # Sets future positions to -inf → softmax makes them 0

    attn_weights = F.softmax(scores, dim=-1)
    # Now each row sums to 1 — a probability distribution

    output = torch.matmul(attn_weights, V)
    # Weighted sum of Values
    return output, attn_weights

### Multi-head attention

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads   # 512/8 = 64 per head

        # Projections for Q, K, V and output
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        # x: (batch, seq_len, d_model)
        batch, seq_len, _ = x.size()
        x = x.view(batch, seq_len, self.n_heads, self.d_k)
        return x.transpose(1, 2)
        # → (batch, n_heads, seq_len, d_k)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_Q(Q))
        K = self.split_heads(self.W_K(K))
        V = self.split_heads(self.W_V(V))

        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        # attn_out: (batch, n_heads, seq_len, d_k)

        # Merge heads back
        batch, _, seq_len, _ = attn_out.size()
        attn_out = attn_out.transpose(1,2).contiguous()
        attn_out = attn_out.view(batch, seq_len, self.d_model)

        return self.W_O(attn_out), weights

### Positional Encoding

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe     = torch.zeros(max_len, d_model)
        pos    = torch.arange(0, max_len).unsqueeze(1).float()
        div    = torch.exp(torch.arange(0, d_model, 2).float()
                           * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)  # even indices
        pe[:, 1::2] = torch.cos(pos * div)  # odd indices

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

### Full Transformer Encoder Block

In [5]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, n_heads)
        self.ffn       = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual
        attn_out, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))  # residual + norm

        # Feed-forward + residual
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))   # residual + norm

        return x

### Full Transformer for text classification (practical use case)

In [6]:
class TransformerClassifier(nn.Module):
    """
    Classifies text into categories.
    Example: classify financial news as Positive/Negative/Neutral
    """
    def __init__(self, vocab_size, d_model=128, n_heads=4,
                 n_layers=2, d_ff=256, max_len=512,
                 num_classes=3, dropout=0.1):
        super().__init__()

        self.embedding   = nn.Embedding(vocab_size, d_model,
                                         padding_idx=0)
        self.pos_enc     = PositionalEncoding(d_model, max_len, dropout)
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.pool       = nn.AdaptiveAvgPool1d(1)   # global average pooling
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (batch, seq_len) token IDs
        x = self.embedding(x)      # (batch, seq_len, d_model)
        x = self.pos_enc(x)        # add positional info

        for layer in self.encoder_layers:
            x = layer(x, mask)     # (batch, seq_len, d_model)

        # Pool across sequence → single vector per sample
        x = x.transpose(1, 2)     # (batch, d_model, seq_len)
        x = self.pool(x).squeeze(-1)   # (batch, d_model)
        x = self.dropout(x)
        return self.classifier(x)  # (batch, num_classes)

# Create model
model = TransformerClassifier(vocab_size=10000, num_classes=3)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

# Test forward pass
sample_input = torch.randint(0, 10000, (4, 64))  # batch=4, seq_len=64
output = model(sample_input)
print(f"Output shape: {output.shape}")  # (4, 3)

Parameters: 1,545,347
Output shape: torch.Size([4, 3])


# Using HuggingFace (production approach)

In [7]:
# In practice you NEVER build Transformers from scratch
# You use pretrained models via HuggingFace

# pip install transformers
from transformers import (AutoTokenizer, AutoModel,
                          pipeline, AutoModelForSequenceClassification)
import torch

# ── 1. Sentiment Analysis with 3 lines ───────────────
classifier = pipeline("sentiment-analysis")
result = classifier("The house prices in Downtown have increased significantly.")
print(result)
# [{'label': 'POSITIVE', 'score': 0.998}]

# ── 2. Get BERT embeddings ─────────────────────────────
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model_bert = AutoModel.from_pretrained('bert-base-uncased')

text   = "House prices depend on location and condition."
tokens = tokenizer(text, return_tensors='pt',
                   padding=True, truncation=True, max_length=128)

with torch.no_grad():
    output = model_bert(**tokens)

# CLS token embedding = sentence-level representation
cls_embedding = output.last_hidden_state[:, 0, :]
print(f"Embedding shape: {cls_embedding.shape}")  # (1, 768)

# ── 3. Fine-tune BERT for classification ─────────────
model_ft = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3  # Positive / Negative / Neutral
)

# Standard PyTorch training loop from Day 11 works here
optimizer = torch.optim.AdamW(model_ft.parameters(), lr=2e-5)
# lr=2e-5 is the standard for fine-tuning BERT — lower than usual

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9986782670021057}]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: torch.Size([1, 768])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
